<a href="https://www.kaggle.com/code/fredericnicholson/give-me-more-cookies?scriptVersionId=239348097" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## Loading and brief analysis 

This notebook implements  https://kaggle.com/competitions/playground-series-s5e5, 2025. Kaggle.

In [ ]:
import numpy as np # linear algebra
import polars as pl # data processing, CSV file I/O (e.g. pd.read_csv)
import polars.selectors as cs
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
train_df = pl.read_csv("/kaggle/input/playground-series-s5e5/train.csv")
test_df = pl.read_csv("/kaggle/input/playground-series-s5e5/test.csv")
sample_submission_df = pl.read_csv("/kaggle/input/playground-series-s5e5/sample_submission.csv")
print (f"{train_df.shape = }, {test_df.shape = }")
with pl.Config (tbl_cols = 20) :
    print (train_df.head(20))

In [ ]:
display (train_df.select(cs.numeric()).describe())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
# Number of columns for the subplot grid
num_cols = 2
features = ["Age", "Height", "Weight", "Duration", "Heart_Rate", "Body_Temp"]


# Calculate the number of rows needed for the subplots
num_rows = (len(features) + num_cols - 1) // num_cols  # Ceiling division



# Create the subplots
fig, axes = plt.subplots(num_rows, num_cols, figsize=(12, 4 * num_rows))  # Adjust figsize as needed

# Flatten the axes array to easily iterate over it
axes = axes.flatten()

# Iterate through the features and create histograms
for i, feature in enumerate(features):
    sns.histplot(data=train_df.to_pandas(), x=feature, ax=axes[i], hue = 'Sex', kde=True, bins = 10)  # Added kde=True for density curve
    axes[i].set_title(f"Distribution of {feature}")
    axes[i].set_xlabel(feature)  # Ensure x-axis label is the feature name
    axes[i].set_ylabel("Frequency")

# Remove any unused subplots (if the number of features is not a multiple of num_cols)
for i in range(len(features), len(axes)):
    fig.delaxes(axes[i])

# Adjust the layout to prevent overlapping titles/labels
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
# # Create the subplots
# fig, axes = plt.subplots(num_rows, num_cols, figsize=(12, 4 * num_rows))  # Adjust figsize as needed

# # Flatten the axes array to easily iterate over it
# axes = axes.flatten()


# # Iterate through the features and create histograms
# for i, feature in enumerate(features):
#     sns.histplot(data=test_df.to_pandas(), x=feature, ax=axes[i], kde=True, bins = 10)  # Added kde=True for density curve
#     axes[i].set_title(f"Distribution of {feature}")
#     axes[i].set_xlabel(feature)  # Ensure x-axis label is the feature name
#     axes[i].set_ylabel("Frequency")

# # Remove any unused subplots (if the number of features is not a multiple of num_cols)
# for i in range(len(features), len(axes)):
#     fig.delaxes(axes[i])

# # Adjust the layout to prevent overlapping titles/labels
# plt.tight_layout()

# # Show the plot
# plt.show()

In [ ]:
gender_map = {"male" : 0,
              "female" : 1}

train_df_mod = train_df.with_columns (pl.col("Sex").replace_strict (gender_map))

train_df_mod = train_df_mod.drop("id")

print (train_df_mod.head())

print (sns.heatmap(train_df_mod.corr().to_pandas()))


In [ ]:
# sns.scatterplot (data = train_df_mod.to_pandas(), x= "Duration", y = "Calories", hue = "Sex")

In [ ]:
# sns.scatterplot (data = train_df_mod.to_pandas(), x= "Heart_Rate", y = "Calories", hue = "Sex")

In [ ]:
# sns.scatterplot (data = train_df_mod.to_pandas(), x= "Body_Temp", y = "Calories", hue = "Sex")

These plots show a significant different distribution for male against female. We accommodate this fact by training two seperate models  

In [ ]:
 train_df = train_df.unique (features + ["Calories"]) 


## feature engineering 

In [ ]:
def remove_outliers (df : pl.DataFrame) -> pl.DataFrame :
    return  df.filter (((pl.col("Calories") / pl.col("Duration") < 11) | (pl.col("Duration") > 22)) & 
                        (pl.col("Calories") < 300))
    

AutoGluon works better with RMSE, so we convert the target to log plus_1  

In [ ]:
from itertools import combinations

gender_map = {"male" : 0,
              "female" : 1}


def add_features (df : pl.DataFrame) -> pl.DataFrame :
    result = df.with_columns (pl.col("Sex").replace_strict (gender_map),
                              (pl.col("Weight")/pl.col("Height")).alias ("BMI"), 
                              ( 30 / pl.Col("Duration")). alias ("my weight"))
giving more weight to shorter workouts 
    
    

    # feature_pairs = list(combinations(features, 2))
    # for feature_pair in feature_pairs :
    #     a,b = feature_pair 
    #     result = result.with_columns ((pl.col(a) * pl.col(b)).log1p().alias (f"{a}_mult_{b}"), 
    #                                  (pl.col(a) / (0.001 + pl.col(b))).log1p().alias (f"{a}_div_{b}"))
    if "Calories" in df.columns :
        result = result.with_columns (pl.col("Calories").log1p().alias ("Calories_log"))
        result = result.drop ("Calories")
    return result         

train_clean = train_df.pipe ( 
                remove_outliers).pipe (
    add_features )

test_clean = test_df.pipe (
    add_features)

with pl.Config (tbl_cols = 20) :
    print (train_clean.head())

print (train_clean.columns)

In [ ]:
train_clean_male = train_clean.filter(pl.col("Sex") ==  0) 
train_clean_female = train_clean.filter(pl.col("Sex") ==  1)
test_clean_male = test_clean.filter(pl.col("Sex") ==  0)
test_clean_female = test_clean.filter(pl.col("Sex") ==  1)

In [ ]:
!pip install ray==2.10.0
!pip install scikit-learn==1.5.2
!pip install autogluon.tabular --no-cache-dir -q
!pip install -U ipywidgets

## Training with AutoGluon

In [ ]:

from autogluon.tabular import TabularPredictor

predictor_male= TabularPredictor(path = '/kaggle/working/Autogluon/male',
                                       label='Calories_log', 
                               problem_type = 'regression', 
                               eval_metric =  'root_mean_squared_error',  
                               sample_weight = 'my_weight',
                               verbosity  = 1,
                               learner_kwargs = {'ignored_columns' : [
                                   'id',
                                   'Sex'
                               #   'my_weight'
                                    ]})



In [ ]:
# train_clean_male1 = train_clean_male.sample (fraction = 0.75, shuffle = True) 
# train_clean_male2 = train_clean_male.join (train_clean_male1, on = "id", how = "anti")

# print (f"{train_clean_male1.shape = },  {train_clean_male2.shape = }")

In [ ]:
predictor_male.fit(train_data= train_clean_male.to_pandas(), 
                        presets= 'best_quality',
    # best_quality, high_quality, medium_quality, 'experimental_quality',                         
                        time_limit = 19000,
                        # num_gpus=1,
#                        raise_on_no_models_fitted = True,
#                        dynamic_stacking=False, 
#                        num_stack_levels=1,
                        #hyperparameters=custom_hyperparameters,
#                         hyperparameters = my_search_hyperparameters  ,
#                         hyperparameter_tune_kwargs=hyperparameter_tune_kwargs,
                        )

In [ ]:
display (predictor_male.leaderboard())

# display (predictor_male.feature_importance (train_clean_male2.to_pandas()))



print  (f"{ predictor_male.predict_oof() = }")

Female distribution looks more linear, so the CV value is expected to be higher 

In [ ]:
predictor_female= TabularPredictor(path = '/kaggle/working/Autogluon/female',
                                       label='Calories_log', 
                                problem_type = 'regression', 
                               eval_metric =  'root_mean_squared_error',  
                               sample_weight = 'my_weight',
                               verbosity  = 1,
                               learner_kwargs = {'ignored_columns' : [
                                    'id',
                                   'Sex' 
                                   
                               #   'my_weight'
                                    ]})

In [ ]:
predictor_female.fit(train_data= train_clean_female.to_pandas(), 
                        presets= 'best_quality',
    # best_quality, high_quality, medium_quality, 'experimental_quality',                         
                        time_limit = 17000,
                        # num_gpus=1,
#                        raise_on_no_models_fitted = True,
#                        dynamic_stacking=False, 
#                        num_stack_levels=1,
                        #hyperparameters=custom_hyperparameters,
#                         hyperparameters = my_search_hyperparameters  ,
#                         hyperparameter_tune_kwargs=hyperparameter_tune_kwargs,
                        )




In [ ]:
display (predictor_female.leaderboard())

print  (f"{ predictor_female.predict_oof() = }")

## Forecasting on Test Dataset

In [ ]:
predictions_male = predictor_male.predict (test_clean_male.to_pandas()) 
print (predictions_male)

In [ ]:
predictions_female = predictor_female.predict (test_clean_female.to_pandas()) 
print (predictions_female)

## Submission

assuming that the target remains within the bounds of the training data, we clip to the min and max values. rounding down should give a slight advantage   

In [ ]:
def convert (pred, test) -> pl.DataFrame :
    converted = pl.Series ("Calories", np.exp(pred) -1) 
    result = test.select ("id")
    result = result.with_columns (converted.alias ("Calories"))
    result = result.with_columns (pl.col("Calories").clip (lower_bound = 1.0, upper_bound = 314.0).floor())
    return result

submission_male = convert (predictions_male, test_clean_male)
submission_female = convert (predictions_female, test_clean_female)

submission = pl.concat ([submission_male, submission_female])

In [ ]:
submission.describe()

In [ ]:
submission.write_csv("submission.csv")
print ("submission complete")

In [ ]:

import zipfile

def move_2_zip (directory_path, archive_name):

    """
    Generated bi Gen AI (Gemini 2.0 flash)
  Zips a directory and all its subdirectories and files into a zip archive.

  Args:
    directory_path: The path to the directory to zip.
    archive_name: The name (including the .zip extension) of the zip archive to create.

  Returns:
    True if the zipping was successful, False otherwise.
    Prints error messages to the console if any issues occur.
  """
    try:
        with zipfile.ZipFile(archive_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
          for root, _, files in os.walk(directory_path):
            for filename in files:
                file_path = os.path.join(root, filename)
                relative_path = os.path.relpath(file_path, directory_path)  # Path relative to the root directory
                try:
                    zipf.write(file_path, relative_path)
                    os.remove(file_path) #Remove file after successful zip
                except Exception as e:
                    print(f"Error zipping/deleting file {filename}: {e}")
                    return False

        print(f"Successfully zipped '{directory_path}' to '{archive_name}' and deleted original files.")
        return True

    except Exception as e:
        print(f"An error occurred: {e}")
        return False


move_2_zip ("/kaggle/working/Autogluon", "/kaggle/working/artifacts")